# Now sending the data from my local postgreSQL that is running on docker to snowflake

In [ ]:
# ! pip install pyspark snowflake-sqlalchemy pandas sqlalchemy numpy

In [ ]:
import pandas as pd
import hashlib
import uuid
from datetime import datetime
import logging
from sqlalchemy import create_engine, text
from snowflake.sqlalchemy import URL
import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# PostgreSQL connection details
POSTGRES_HOST = "postgres-local"
POSTGRES_PORT = "5432"
POSTGRES_DATABASE = "data_platform_db"
POSTGRES_USER = "source_user"
POSTGRES_PASSWORD = "source_pass"

# Snowflake connection details
SNOWFLAKE_ACCOUNT = 'KBBXLLH-XQ81200'
SNOWFLAKE_USER = 'Ibrahimhegazi'
SNOWFLAKE_PASSWORD = '!!ASDFqwer1234@@'
SNOWFLAKE_DATABASE = 'SALES_OPS_DB'
SNOWFLAKE_SCHEMA = 'BRONZE'
SNOWFLAKE_WAREHOUSE = 'BRONZE_ODS'
SNOWFLAKE_ROLE = 'BRONZE_ETL_ROLE'

# Create PostgreSQL engine
POSTGRES_ENGINE = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}')

# Create Snowflake engine
snowflake_url = URL(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA,
    warehouse=SNOWFLAKE_WAREHOUSE,
    role=SNOWFLAKE_ROLE
)
SNOWFLAKE_ENGINE = create_engine(snowflake_url)

# Generate a unique load_id for this batch
LOAD_ID = f"load_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Table list with their source table names and key columns for hash generation
TABLES = {
    'region': {
        'key_columns': ['r_regionkey'],
        'business_keys': ['r_regionkey']
    },
    'nation': {
        'key_columns': ['n_nationkey'],
        'business_keys': ['n_nationkey']
    },
    'part': {
        'key_columns': ['p_partkey'],
        'business_keys': ['p_partkey']
    },
    'supplier': {
        'key_columns': ['s_suppkey'],
        'business_keys': ['s_suppkey']
    },
    'customer': {
        'key_columns': ['c_custkey'],
        'business_keys': ['c_custkey']
    },
    'orders': {
        'key_columns': ['o_orderkey'],
        'business_keys': ['o_orderkey']
    },
    'partsupp': {
        'key_columns': ['ps_id'],
        'business_keys': ['ps_partkey', 'ps_suppkey']
    },
    'lineitem': {
        'key_columns': ['l_id'],
        'business_keys': ['l_orderkey', 'l_linenumber']
    }
}

def generate_record_hash(row, business_keys):
    """Generate SHA-256 hash based on business keys"""
    hash_strings = []
    for key in business_keys:
        value = str(row.get(key, 'NULL')) if row.get(key) is not None else 'NULL'
        hash_strings.append(value)
    
    concatenated = "|".join(hash_strings)
    return hashlib.sha256(concatenated.encode()).hexdigest()

def add_lineage_columns(df, source_table, business_keys):
    """Add lineage columns to DataFrame"""
    logger.info(f"Adding lineage columns for table: {source_table}")
    
    current_time = datetime.now()
    
    # Add lineage columns
    df['_source_table'] = source_table
    df['_last_update_time'] = current_time
    df['_processed_at'] = current_time
    df['_loaded_at'] = current_time
    df['_is_duplicate'] = False
    df['_error_flag'] = False
    df['_error_message'] = None
    df['_load_id'] = LOAD_ID
    
    # Generate record hash for each row
    if business_keys:
        df['_record_hash'] = df.apply(lambda row: generate_record_hash(row, business_keys), axis=1)
    else:
        # Hash all columns if no business keys specified
        df['_record_hash'] = df.apply(lambda row: generate_record_hash(row, df.columns.tolist()), axis=1)
    
    logger.info(f"Added {len(df.columns)} columns total")
    return df

def extract_table_pandas(table_name):
    """Extract data from PostgreSQL using pandas"""
    logger.info(f"Extracting data from PostgreSQL table: bronze.{table_name}")
    
    try:
        query = f"SELECT * FROM bronze.{table_name}"
        df = pd.read_sql(query, POSTGRES_ENGINE)
        
        record_count = len(df)
        logger.info(f"Extracted {record_count} records from {table_name}")
        
        return df
        
    except Exception as e:
        logger.error(f"Failed to extract {table_name}: {str(e)}")
        raise

def extract_table_in_batches(table_name, batch_size=50000):
    """Extract data in batches using server-side cursor for large tables"""
    logger.info(f"Extracting data from PostgreSQL table: bronze.{table_name} in batches")
    
    # Get total count first
    count_query = f"SELECT COUNT(*) as count FROM bronze.{table_name}"
    total_count = pd.read_sql(count_query, POSTGRES_ENGINE)['count'].iloc[0]
    logger.info(f"Total records in {table_name}: {total_count}")
    
    # Use server-side cursor for large tables
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DATABASE,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    
    try:
        # Create a server-side cursor
        cur = conn.cursor(name=f'cursor_{table_name}', cursor_factory=RealDictCursor)
        cur.execute(f"SELECT * FROM bronze.{table_name}")
        
        batch_count = 0
        total_records = 0
        
        while True:
            rows = cur.fetchmany(batch_size)
            if not rows:
                break
            
            # Convert to DataFrame
            batch_df = pd.DataFrame(rows)
            batch_count += 1
            total_records += len(batch_df)
            
            logger.info(f"  Extracted batch {batch_count} ({len(batch_df)} records)")
            
            # Yield the batch for processing
            yield batch_df
        
        logger.info(f"Extracted total {total_records} records from {table_name} in {batch_count} batches")
        
    finally:
        cur.close()
        conn.close()

def validate_data_quality(df, table_name):
    """Basic data quality checks"""
    config = TABLES[table_name]
    key_columns = config.get('key_columns', [])
    
    if not key_columns:
        return df
    
    error_messages = []
    
    for key_col in key_columns:
        if key_col in df.columns:
            null_count = df[key_col].isnull().sum()
            if null_count > 0:
                error_msg = f"NULL value in {key_col}"
                error_messages.append(error_msg)
                logger.warning(f"Found {null_count} NULL values in {key_col} for {table_name}")
    
    # Mark error records
    if error_messages:
        df['_error_flag'] = df[key_columns].isnull().any(axis=1)
        df['_error_message'] = df[key_columns].isnull().apply(
            lambda x: "; ".join(error_messages) if x else None
        )
    else:
        df['_error_flag'] = False
        df['_error_message'] = None
    
    return df

def deduplicate_within_batch(df, table_name, business_keys):
    """Mark duplicates within the current batch"""
    if not business_keys:
        return df
    
    valid_keys = [key for key in business_keys if key in df.columns]
    
    if not valid_keys:
        return df
    
    logger.info(f"Checking for duplicates in {table_name} batch using keys: {valid_keys}")
    
    # Mark duplicates (keep first occurrence, mark others)
    df['_is_duplicate'] = df.duplicated(subset=valid_keys, keep='first')
    
    duplicate_count = df['_is_duplicate'].sum()
    if duplicate_count > 0:
        logger.info(f"Found {duplicate_count} duplicate records in batch")
    
    return df

def load_to_snowflake_batched(df, table_name, batch_size=10000):
    """Load DataFrame to Snowflake in batches"""
    logger.info(f"Loading data to Snowflake table: {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA}.{table_name}")
    
    # Prepare data for Snowflake
    for col_name in df.columns:
        # Convert timestamps to string format
        if 'time' in col_name.lower() or 'date' in col_name.lower():
            if pd.api.types.is_datetime64_any_dtype(df[col_name]):
                df[col_name] = df[col_name].dt.strftime('%Y-%m-%d %H:%M:%S')
        
        # Handle NaN/None values
        df[col_name] = df[col_name].where(pd.notna(df[col_name]), None)
    
    total_rows = len(df)
    logger.info(f"Loading {total_rows} records in batches of {batch_size}")
    
    for i in range(0, total_rows, batch_size):
        batch = df.iloc[i:i+batch_size]
        try:
            batch.to_sql(
                table_name,
                SNOWFLAKE_ENGINE,
                schema=SNOWFLAKE_SCHEMA,
                if_exists='append',
                index=False,
                method='multi',
                chunksize=min(batch_size, 5000)  # Snowflake works better with smaller chunks
            )
            logger.info(f"  Loaded batch {i//batch_size + 1}/{(total_rows + batch_size - 1)//batch_size} ({len(batch)} records)")
        except Exception as e:
            logger.error(f"Failed to load batch {i//batch_size + 1}: {str(e)}")
            # Try to identify problematic records
            logger.info("Attempting to identify problematic records...")
            for idx, row in batch.iterrows():
                try:
                    pd.DataFrame([row]).to_sql(
                        table_name,
                        SNOWFLAKE_ENGINE,
                        schema=SNOWFLAKE_SCHEMA,
                        if_exists='append',
                        index=False
                    )
                except Exception as row_error:
                    logger.error(f"Failed to load row {idx}: {row_error}")
                    logger.error(f"Problematic row data: {row.to_dict()}")
            raise
    
    logger.info(f"Successfully loaded all {total_rows} records to {table_name}")

def process_table_small(table_name):
    """Process small tables (region, nation) in one go"""
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing small table: {table_name}")
    logger.info(f"{'='*60}")
    
    try:
        # Step 1: Extract from PostgreSQL
        df = extract_table_pandas(table_name)
        
        if len(df) == 0:
            logger.info(f"No data found in {table_name}, skipping...")
            return True
        
        # Step 2: Add lineage columns
        config = TABLES[table_name]
        df = add_lineage_columns(df, table_name, config.get('business_keys', []))
        
        # Step 3: Data quality validation
        df = validate_data_quality(df, table_name)
        
        # Step 4: Check for duplicates within batch
        if config.get('business_keys'):
            df = deduplicate_within_batch(df, table_name, config['business_keys'])
        
        # Step 5: Load to Snowflake
        load_to_snowflake_batched(df, table_name, batch_size=10000)
        
        # Log summary
        error_count = df['_error_flag'].sum()
        duplicate_count = df['_is_duplicate'].sum()
        
        logger.info(f"✅ Successfully processed {table_name}")
        logger.info(f"   Total records: {len(df)}")
        logger.info(f"   Error records: {error_count}")
        logger.info(f"   Duplicate records: {duplicate_count}")
        logger.info(f"   Load ID: {LOAD_ID}")
        
        return True
        
    except Exception as e:
        logger.error(f"❌ Failed to process {table_name}: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return False

def process_table_large(table_name):
    """Process large tables in batches"""
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing large table: {table_name}")
    logger.info(f"{'='*60}")
    
    try:
        config = TABLES[table_name]
        total_batches = 0
        total_records = 0
        total_errors = 0
        total_duplicates = 0
        
        # Process in batches
        for batch_df in extract_table_in_batches(table_name, batch_size=50000):
            if len(batch_df) == 0:
                continue
            
            # Step 2: Add lineage columns
            batch_df = add_lineage_columns(batch_df, table_name, config.get('business_keys', []))
            
            # Step 3: Data quality validation
            batch_df = validate_data_quality(batch_df, table_name)
            
            # Step 4: Check for duplicates within batch
            if config.get('business_keys'):
                batch_df = deduplicate_within_batch(batch_df, table_name, config['business_keys'])
            
            # Step 5: Load to Snowflake
            load_to_snowflake_batched(batch_df, table_name, batch_size=10000)
            
            # Track stats
            total_batches += 1
            total_records += len(batch_df)
            total_errors += batch_df['_error_flag'].sum()
            total_duplicates += batch_df['_is_duplicate'].sum()
            
            logger.info(f"  Batch {total_batches} complete: {len(batch_df)} records")
        
        logger.info(f"✅ Successfully processed {table_name}")
        logger.info(f"   Total records: {total_records}")
        logger.info(f"   Total batches: {total_batches}")
        logger.info(f"   Error records: {total_errors}")
        logger.info(f"   Duplicate records: {total_duplicates}")
        logger.info(f"   Load ID: {LOAD_ID}")
        
        return True
        
    except Exception as e:
        logger.error(f"❌ Failed to process {table_name}: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return False

def main():
    """Main execution function"""
    logger.info("=" * 60)
    logger.info("Starting Bronze to Snowflake Migration")
    logger.info(f"Load ID: {LOAD_ID}")
    logger.info("=" * 60)
    
    # Small tables (process in one go)
    small_tables = ['region', 'nation']
    
    # Large tables (process in batches)
    large_tables = ['part', 'supplier', 'customer', 'orders', 'partsupp', 'lineitem']
    
    success_count = 0
    failed_tables = []
    
    # Process small tables
    for table_name in small_tables:
        if process_table_small(table_name):
            success_count += 1
        else:
            failed_tables.append(table_name)
    
    # Process large tables
    for table_name in large_tables:
        if process_table_large(table_name):
            success_count += 1
        else:
            failed_tables.append(table_name)
    
    # Summary
    logger.info("\n" + "=" * 60)
    logger.info("Migration Summary")
    logger.info("=" * 60)
    logger.info(f"Total tables: {len(small_tables) + len(large_tables)}")
    logger.info(f"Successfully processed: {success_count}")
    logger.info(f"Failed: {len(failed_tables)}")
    
    if failed_tables:
        logger.info(f"Failed tables: {', '.join(failed_tables)}")
    
    logger.info(f"Load ID: {LOAD_ID} - use this for tracing")
    logger.info("=" * 60)

if __name__ == "__main__":
    main()

2026-03-30 20:02:05,975 - INFO - ============================================================
2026-03-30 20:02:05,976 - INFO - Starting Bronze to Snowflake Migration
2026-03-30 20:02:05,977 - INFO - Load ID: load_20260330_200205_fe358f9a
2026-03-30 20:02:05,978 - INFO - ============================================================
2026-03-30 20:02:05,978 - INFO - 
2026-03-30 20:02:05,979 - INFO - Processing small table: region
2026-03-30 20:02:05,980 - INFO - ============================================================
2026-03-30 20:02:05,981 - INFO - Extracting data from PostgreSQL table: bronze.region
2026-03-30 20:02:06,058 - INFO - Extracted 5 records from region
2026-03-30 20:02:06,059 - INFO - Adding lineage columns for table: region
2026-03-30 20:02:06,066 - INFO - Added 12 columns total
2026-03-30 20:02:06,068 - INFO - Checking for duplicates in region batch using keys: ['r_regionkey']
2026-03-30 20:02:06,071 - INFO - Loading data to Snowflake table: SALES_OPS_DB.BRONZE.region
2

# Run this after migration to verify data

In [ ]:
from sqlalchemy import create_engine, text
from snowflake.sqlalchemy import URL
import pandas as pd

snowflake_url = URL(
    account='KBBXLLH-XQ81200',
    user='Ibrahimhegazi',
    password='!!ASDFqwer1234@@',
    database='SALES_OPS_DB',
    schema='BRONZE',
    warehouse='BRONZE_ODS',
    role='BRONZE_ETL_ROLE'
)

engine = create_engine(snowflake_url)

# Verify lineage columns are populated
with engine.connect() as conn:
    # Check customer table
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total_records,
            COUNT(_record_hash) as has_hash,
            COUNT(_load_id) as has_load_id,
            COUNT(DISTINCT _load_id) as distinct_load_ids,
            MIN(_loaded_at) as earliest_load,
            MAX(_loaded_at) as latest_load
        FROM BRONZE.customer
    """))
    
    for row in result:
        print(f"Customer Table Statistics:")
        print(f"  Total Records: {row.total_records}")
        print(f"  Records with Hash: {row.has_hash}")
        print(f"  Records with Load ID: {row.has_load_id}")
        print(f"  Distinct Load IDs: {row.distinct_load_ids}")
        print(f"  Load Date Range: {row.earliest_load} to {row.latest_load}")
    
    # Check for any error records
    result = conn.execute(text("""
        SELECT 
            _source_table,
            COUNT(*) as error_count,
            MAX(_error_message) as sample_error
        FROM BRONZE.customer
        WHERE _error_flag = TRUE
        GROUP BY _source_table
    """))
    
    for row in result:
        print(f"\nError Records in {row._source_table}: {row.error_count}")
        if row.sample_error:
            print(f"  Sample Error: {row.sample_error}")